# IndicTrans2: four-stage corpus-quality evaluation

**Default Stage 1:** untouched model, full development and test inference, evaluation, figures and independent archive. No finetuning. Use the same frozen corpus splits across model families. Unlike NLLB, IndicTrans2 requires the official `IndicTransToolkit` and explicit review/opt-in to custom model code. See `docs/four-stage-workflow.md`.


## 1. Experiment settings — only editable cell

Intermediate checkpoints now default to scratch storage with model-only saves and minimal retention. To back up automatically to Google Drive, configure `checkpoint_backup` and its Drive credentials/mount in the **same first cell**. See `docs/checkpoint-storage.md` before changing these controls.


In [ ]:
# ONE EDITABLE CELL. Reimport notebook/restart runtime for a new experiment.
EXPERIMENT = {
    # STAGES: lowest_baseline (default) | low_baseline | decoding_search | language_extension
    "stage": "lowest_baseline",
    "run_label": "corpus-study-1",

    # DATA: two aligned files; or one tsv/jsonl; or pre_split/example.
    "data_mode": "aligned_text",
    "source_lang": "mai_Deva",  # Use genuinely supported IT2 tags for Stages 1–3.
    "target_lang": "hin_Deva",
    "source_text": "/kaggle/input/YOUR_DATASET/mai.txt",
    "target_text": "/kaggle/input/YOUR_DATASET/hin.txt",
    "tsv_file": "/kaggle/input/YOUR_DATASET/parallel.tsv",
    "tsv_has_header": False,
    "parallel_jsonl": "data/local/mai-hin/parallel.jsonl",
    "split_directory": "data/local/mai-hin/splits",
    "dataset_id": "corpus-v1",
    "split_seed": 42, "split_ratios": (0.80, 0.10, 0.10),
    "group_by_document": True,
    "validate_data": False,
    "reject_within_split_duplicates": False,  # Allow repeated corpus pairs; ALWAYS block cross-split leakage.     # Full corpus audit is optional.
    "data_report": False,       # Extra data diagnostics (separate from score plots).

    # MODEL: upstream IndicTrans2 requires separately installed IndicTransToolkit.
    "model_id": None,           # None resolves official Indic-to-Indic distilled model.
    "conditioning": "native",
    "allow_remote_code": False, # Review upstream HF custom code, then set True.
    "proxy_source_lang": None, "proxy_target_lang": None,
    "beams": 4, "length_penalty": 1.0,
    "max_new_tokens": 128, "max_length": 128, "batch_size": 1,

    # STAGE 2/4: full finetuning (not a 1-step smoke test).
    "allow_training": False,    # True only for low_baseline/language_extension.
    "finetune_mode": "full",     # Switch to lora when GPU cannot fit full model.
    "train_seed": 42, "epochs": 3, "max_steps": -1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-5, "precision": "fp16",
    "gradient_checkpointing": True, "lora_rank": 8,

    # STAGE 3: select decoding on development set, test once after selection.
    "search_source_stage": "low_baseline", "search_metric": "chrfpp",
    "search_grid": [
        {"beams": 1, "length_penalty": 1.0},
        {"beams": 4, "length_penalty": 1.0},
        {"beams": 6, "length_penalty": 1.0},
    ],

    # STAGE 4: use dedicated_token and supported processing proxies if necessary.
    # Adding new tokens requires full finetuning; not claimed native IT2 coverage.
    "result_plots": True, "save_archive": True,
    "include_checkpoints": False,
    "setup_mode": "train",      # GPU environment, plus install/review IndicTransToolkit.
    # CHECKPOINT STORAGE (Stage 2/4, full finetuning).
    "checkpoint_storage": "auto", # Kaggle: /kaggle/tmp; Colab: /content.
    "checkpoint_limit": 1,        # Minimum retention; best model may need 2 saves.
    "checkpoint_model_only": True,# Model-only saves: far smaller, NOT exact resume.
    "checkpoint_min_free_gib": 5,  # Fail early if scratch is nearly full.
    "checkpoint_cleanup_on_success": True,
    # Optional external backup. Kaggle: rclone secret; Colab: mounted_directory.
    "checkpoint_backup": "none",   # none | rclone | mounted_directory
    "checkpoint_rclone_remote": "drive:BhaashikMT/checkpoints",
    "checkpoint_rclone_secret": None, # Kaggle Secret name storing rclone.conf.
    "checkpoint_backup_directory": "/content/drive/MyDrive/BhaashikMT/checkpoints",
    "refresh_repo": False,

    # OPTIONAL CLEANUP. Leave "off" normally. Run ONLY settings+cleanup cells to use.
    "cleanup_action": "off",        # off | preview | delete
    "cleanup_target_stage": "low_baseline", # Stage to clean, even if currently in another stage.
    "cleanup_confirmation": "",    # Exact phrase printed by preview; required for delete.
}


## 2. Setup

Select GPU and Internet; install reviewed IndicTransToolkit separately.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = (Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir() else Path.cwd().resolve())
os.environ["USE_TF"], os.environ["USE_FLAX"] = "0", "0"
if not (ROOT / "pyproject.toml").is_file():
    if ROOT.exists(): raise ValueError(f"Repository checkout is incomplete: {ROOT}")
    subprocess.run(["git", "clone", "https://github.com/bhaashik/BhaashikMTFinetuning.git",
                    str(ROOT)], check=True)
elif EXPERIMENT["refresh_repo"]:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if any(m in sys.modules for m in ("torch","transformers","peft","datasets")):
    raise RuntimeError("Restart the notebook kernel before installing model packages.")
cmd = [sys.executable, str(ROOT / "scripts" / "setup_notebook.py"),
       "--extras", EXPERIMENT["setup_mode"]]
if EXPERIMENT["setup_mode"].startswith("train"):
    cmd.append("--require-cuda")
subprocess.run(cmd, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="indictrans2")
print("Repo:", ROOT)
# IMPORTANT: IndicTransToolkit is an ADDITIONAL dependency. Install its
# official reviewed version in this fresh session before model inference/training.
# See docs/experiment-guide.md; avoid mixing incompatible model stacks.


## 3. Input conversion and frozen splits


In [ ]:
from bhaashik.experiment_workflow import (
    prepare_parallel_data, experiment_paths, training_parameters,
    write_experiment_config, report_data_if_enabled,
)
PREPARED = prepare_parallel_data(ROOT, EXPERIMENT, model_family="indictrans2")
PATHS = experiment_paths(ROOT, EXPERIMENT, family="indictrans2", prepared=PREPARED)
STAGE_DIR = PATHS["stage"]
report_data_if_enabled(EXPERIMENT, PREPARED, PATHS["report"])
TRAIN_CONFIG = training_parameters(EXPERIMENT, PREPARED, PATHS,
                                   model_family="indictrans2")
TRAIN_CONFIG_PATH = write_experiment_config(
    EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG)
print("Stage:", EXPERIMENT["stage"], "| Files:", STAGE_DIR)


## 4. Stage summary


In [ ]:
from bhaashik.indictrans2_adaptation import processing_pair, _model_id
print("Stage:", EXPERIMENT["stage"])
print("Official model:", EXPERIMENT["model_id"] or _model_id(
    EXPERIMENT["source_lang"], EXPERIMENT["target_lang"]))
print("No training in Stage 1/3.")
print("Stage 2/4 computes development loss each epoch.")


## 5. Model support / processing checks


In [ ]:
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="indictrans2")
from bhaashik.indictrans2_adaptation import processing_pair
print("Processor language route:", processing_pair(
    EXPERIMENT["source_lang"], EXPERIMENT["target_lang"],
    strategy=EXPERIMENT["conditioning"],
    proxy_src=EXPERIMENT["proxy_source_lang"],
    proxy_tgt=EXPERIMENT["proxy_target_lang"]))
print("Data counts:", PREPARED["counts"])
if not EXPERIMENT["allow_remote_code"]:
    print("REQUIRED for inference: inspect upstream HF model code, install IndicTransToolkit,")
    print("then set allow_remote_code=True in the first cell.")


## 6. Run stage and save outputs

Independent of a successful finetuning checkpoint.


In [ ]:
from bhaashik.staged_experiments import run_stage
# Stages 1/2/4: full dev+test inference, metrics, plots and stage ZIP.
# Stage 3: choose decoding only from dev, then evaluate selected choice on test.
RESULT = run_stage(ROOT, EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG_PATH,
                   family="indictrans2")
print("Complete:", RESULT["stage"])


## 7. Inspect scores and predictions


In [ ]:
import json
if (STAGE_DIR / "stage_results.json").is_file():
    results = json.loads((STAGE_DIR / "stage_results.json").read_text())
    print("DEV:", results["dev_scores"])
    print("TEST:", results["test_scores"])
    print("Predictions:", results["predictions"])
else:
    print("Inspect stage_manifest.json for partial results or setup issues.")


## 8. Archive path

Automatically saved by runner on completion or failure.


In [ ]:
print("Stage files:", STAGE_DIR)
if PATHS["archive"].is_file():
    print("Preserve:", PATHS["archive"])
else:
    print("No stage ZIP. Check save_archive and stage_manifest.json.")


## Optional cleanup of a mistaken experiment

**Run this cell separately, not as part of the normal experiment.** In the
first settings cell set `cleanup_action="preview"` and
`cleanup_target_stage` to the mistaken stage. Run only this cleanup
cell to see its files. For deletion, change `cleanup_action="delete"`
and paste the exact printed phrase into `cleanup_confirmation`, then
run this cell alone. It cannot delete source data, split manifests,
other stages, or shared report folders. Default is off.


In [ ]:
# OPTIONAL: clean an incorrect generated experiment, without touching source data.
# To clean: set cleanup_action="preview" in the FIRST settings cell and run
# this cell ONLY (after the settings cell); read the files it lists.
# To delete: set cleanup_action="delete", copy its exact confirmation phrase
# into cleanup_confirmation, and run THIS CELL ONLY. Do not click Run All.
# The selected stage, its ZIP and stage-specific configs are the only targets.
from pathlib import Path
import sys

if EXPERIMENT["cleanup_action"] == "off":
    print("Cleanup disabled. To use it, edit cleanup_* in the first settings cell.")
else:
    root_for_cleanup = (
        Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir()
        else Path.cwd()
    )
    if str(root_for_cleanup / "src") not in sys.path:
        sys.path.insert(0, str(root_for_cleanup / "src"))
    from bhaashik.experiment_cleanup import cleanup_stage
    cleanup_plan = cleanup_stage(
        root_for_cleanup, EXPERIMENT,
        family="indictrans2",
        stage=EXPERIMENT["cleanup_target_stage"],
        action=EXPERIMENT["cleanup_action"],
        confirmation=EXPERIMENT["cleanup_confirmation"],
    )
    print("Deletion completed:", cleanup_plan["deleted"])
